In [1]:
import os
import time
import string

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates

from tqdm import tqdm

# Font
from matplotlib import font_manager
font_path = "/workspace/KENTECH/fonts/"
font_list = os.listdir(font_path)
for font_file in font_list:
    try:
        font_manager.fontManager.addfont(font_path + font_file)
    except:
        raise Exception(f"Cannot Load {font_path+font_file}")

plt.rcParams["figure.dpi"] = 300

In [142]:
from KISTI_DB_Manager import manage, preview
import imp
imp.reload(manage), imp.reload(preview)

(<module 'KISTI_DB_Manager.manage' from '/workspace/SQL/KISTI_DB_Manager/manage.py'>,
 <module 'KISTI_DB_Manager.preview' from '/workspace/SQL/KISTI_DB_Manager/preview.py'>)

In [4]:
Extra_ratio = 1.2
Min_Year = 1900
Max_Year = 2100
PATH = '../Data/Mobility/CWTS_202401/Footprint data/'
SEP = '\t'

flist = sorted([x for x in os.listdir(PATH) if 'txt' in x])

# Make Description

In [131]:
for f in tqdm(flist):
    df = pd.read_csv(f'{PATH}{f}', sep=SEP)
    df_res = pd.DataFrame([])
    for i, col in enumerate(df.columns):
        _series = df[col]
        res = preview.get_Table_Description(_series)
        df_res[col] = res
    # break
    df_res.T.to_csv(f'{PATH}Desc_{f[:-4]}.csv')

100%|█████████████████████████████████████████████| 7/7 [00:22<00:00,  3.15s/it]


In [118]:
df_res.T

,Description,Type,Example,Coverage,min,max,mean,std,top,uniq,freq,entr,Failed,_max_range,uniq_ratio,Null_ratio,is_key
researcher_id,,VARCHAR(32),ur.01044101146.20,1.0,15,18,NaN,NaN,ur.0707230301.15,177031,0.00003,17.329308,False,27,0.751682,0,True
yfp,,YEAR,2005,1.0,2005,2020,2010.87419,4.633613,NaN,16,0.17743,3.855809,False,2100,0.000068,0,False
ylp,,YEAR,2021,1.0,2006,2021,2018.339794,3.569388,NaN,16,0.444311,2.921405,False,2100,0.000068,0,False
yfa,,YEAR,2012,1.0,2005,2021,2012.031637,4.693709,NaN,17,0.112525,4.006538,False,2100,0.000072,0,False
entity,,VARCHAR(16),PRI_NST,1.0,3,9,NaN,NaN,UNIV_UNIV,9,0.623155,1.847171,False,13,0.000038,0,False
type,,VARCHAR(1),N,1.0,1,1,NaN,NaN,O,2,0.744795,0.819423,False,1,0.000008,0,False
years_active,,TINYINT UNSIGNED,17,1.0,2,17,8.465605,4.951192,NaN,16,0.105081,3.910714,False,254,0.000068,0,False
footprint,,VARCHAR(32),bgggbgbff00000g00,1.0,2,17,NaN,NaN,11,84707,0.073886,12.351414,False,25,0.35967,0,False
n_entity,,TINYINT UNSIGNED,3,1.0,1,7,1.550534,0.655558,NaN,7,0.527355,1.321514,False,254,0.00003,0,False
mobile_across_entity,,BOOLEAN,1,1.0,0,1,0.472645,0.499252,NaN,2,0.527355,0.99784,False,1,0.000008,0,False


# Handling the DB

In [12]:
import pymysql

from KISTI_DB_Manager import manage, preview
import imp
imp.reload(manage), imp.reload(preview)

(<module 'KISTI_DB_Manager.manage' from '/workspace/SQL/KISTI_DB_Manager/manage.py'>,
 <module 'KISTI_DB_Manager.preview' from '/workspace/SQL/KISTI_DB_Manager/preview.py'>)

In [13]:
db_config = {
    'host': 'localhost',  # Update as needed
    'user': 'user',       # Update as needed
    'password': '1234',       # Update as needed
    'database': 'CWTS_Footprints_20230105'  # Update as needed
}

data_config = {
    'PATH': PATH,
    'file_name': flist[0],
    'SEP': SEP,
    'table_name': 'table_name',
    'out_path': '../Data/SQL/',
}

Port = 0 # Port for DB with host
CHARACTER_SET = 'utf8mb4'
COLLATE = 'utf8mb4_unicode_520_ci'

In [ ]:
manage.drop_DB(db_config['database'], db_config)
manage.create_DB(db_config['database'], CHARACTER_SET, COLLATE, db_config)

# Table name
for f in tqdm(flist):
    data_config['file_name'] = f
    data_config['table_name'] = f[:-4]
    
    # Generate and execute CREATE TABLE SQL
    manage.create_table(data_config, db_config)
    manage.fill_table_from_file(data_config, db_config)
    manage.set_index(db_config, data_config)
    manage.optimize_table(db_config, data_config)

Database `CWTS_Footprints_20230105` dropped successfully.
Database `CWTS_Footprints_20230105` created successfully.


  0%|                                                     | 0/7 [00:00<?, ?it/s]

Table `Footprints_cities_20230105` created successfully.
Data inserted into table `Footprints_cities_20230105` successfully.
Set Index the researcher_id on `Footprints_cities_20230105` successfully.


 14%|██████▍                                      | 1/7 [00:31<03:07, 31.32s/it]

Optimize table `Footprints_cities_20230105` successfully.
Table `Footprints_countries_20230105` created successfully.
Data inserted into table `Footprints_countries_20230105` successfully.


 29%|████████████▊                                | 2/7 [00:54<02:12, 26.49s/it]

Optimize table `Footprints_countries_20230105` successfully.
Table `Footprints_org+cities_20230105` created successfully.
Data inserted into table `Footprints_org+cities_20230105` successfully.


 43%|███████████████████▎                         | 3/7 [01:30<02:02, 30.65s/it]

Optimize table `Footprints_org+cities_20230105` successfully.
Table `Footprints_organizations_20230105` created successfully.


In [ ]:
manage.backup_database_subprocess(db_config, data_config)